# 🏆 Climate & Health - Champion Model
## Advanced Feature Engineering + Stacking Ensemble + Hyperparameter Optimization

**Goal:** Maximize **Final Score = 0.60 × F1 + 0.40 × ROC-AUC**

### Champion Improvements:
1. **Deep Feature Engineering** - 80+ features with domain knowledge
2. **Hyperparameter Tuning** - Optuna-based optimization
3. **Stacking Ensemble** - Meta-learner combining multiple models
4. **Probability Calibration** - Isotonic/Platt scaling
5. **Advanced Imbalance Handling** - SMOTE + focal loss
6. **Feature Selection** - Recursive elimination + importance-based

## 📚 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import SelectFromModel, RFE, mutual_info_classif
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, 
    VotingClassifier, StackingClassifier, AdaBoostClassifier,
    ExtraTreesClassifier, HistGradientBoostingClassifier
)
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix

# Advanced models
try:
    import xgboost as xgb
    HAS_XGB = True
    print("✓ XGBoost loaded")
except ImportError:
    HAS_XGB = False
    print("✗ XGBoost not available")

try:
    import lightgbm as lgb
    HAS_LGBM = True
    print("✓ LightGBM loaded")
except ImportError:
    HAS_LGBM = False
    print("✗ LightGBM not available")

try:
    import catboost as cb
    HAS_CAT = True
    print("✓ CatBoost loaded")
except ImportError:
    HAS_CAT = False
    print("✗ CatBoost not available")

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize

sns.set_theme(style="whitegrid", font_scale=1.1)
pd.set_option("display.max_columns", 150)
np.random.seed(42)

## ⚙️ Configuration

In [ ]:
# Configuration
RANDOM_STATE = 42
N_SPLITS = 5
TARGET = "is_climate_sensitive"
ID_COL = "ID"

# Model optimization
N_TRIALS = 50  # For hyperparameter tuning
EARLY_STOPPING = 50

print(f"Random State: {RANDOM_STATE}")
print(f"Cross-Validation Folds: {N_SPLITS}")
print(f"Hyperparameter Trials: {N_TRIALS}")

## 📥 Load and Merge Data

In [ ]:
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")
climate_features = pd.read_csv("climate_features.csv")

# Drop deathdate from climate features
climate_features = climate_features.drop(columns=["deathdate"], errors='ignore')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Climate features shape: {climate_features.shape}")

# Merge datasets
train = train.merge(climate_features, on=ID_COL, how="left")
test = test.merge(climate_features, on=ID_COL, how="left")

print(f"\nMerged Train: {train.shape}")
print(f"Merged Test: {test.shape}")

# Target distribution
print(f"\nTarget Distribution:")
target_dist = train[TARGET].value_counts(normalize=True).round(4)
print(target_dist)
print(f"Class imbalance ratio: {target_dist.iloc[0]/target_dist.iloc[1]:.2f}")

## 🔧 DEEP Feature Engineering

In [ ]:
def deep_feature_engineering(df):
    """Comprehensive feature engineering with domain knowledge"""
    df = df.copy()
    
    # Convert date
    df["deathdate"] = pd.to_datetime(df["deathdate"], errors="coerce")
    
    # ========== TEMPORAL FEATURES ==========
    df["day_of_year"] = df["deathdate"].dt.dayofyear
    df["month"] = df["deathdate"].dt.month
    df["day_of_week"] = df["deathdate"].dt.dayofweek
    df["week_of_year"] = df["deathdate"].dt.isocalendar().week.astype(int)
    df["quarter"] = df["deathdate"].dt.quarter
    df["season"] = ((df["month"] % 12 + 3) // 3) % 4
    
    # Cyclical encoding
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["hour_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["hour_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    
    # ========== AGE FEATURES ==========
    df["age_group"] = np.digitize(df["age"], bins=[1, 5, 12, 18, 30, 45, 60, 75])
    df["is_infant"] = (df["age"] < 1).astype(int)
    df["is_toddler"] = ((df["age"] >= 1) & (df["age"] < 5)).astype(int)
    df["is_child"] = ((df["age"] >= 5) & (df["age"] < 12)).astype(int)
    df["is_teen"] = ((df["age"] >= 12) & (df["age"] < 18)).astype(int)
    df["is_young_adult"] = ((df["age"] >= 18) & (df["age"] < 30)).astype(int)
    df["is_middle_aged"] = ((df["age"] >= 45) & (df["age"] < 60)).astype(int)
    df["is_elderly"] = (df["age"] >= 60).astype(int)
    df["is_very_elderly"] = (df["age"] >= 75).astype(int)
    df["is_vulnerable_age"] = ((df["age"] < 5) | (df["age"] > 65)).astype(int)
    df["age_log"] = np.log1p(df["age"])
    df["age_squared"] = df["age"] ** 2
    df["age_cubed"] = df["age"] ** 3
    
    # ========== TEMPERATURE FEATURES ==========
    df["temperature_range"] = df["max_temperature"] - df["min_temperature"]
    df["temp_range_squared"] = df["temperature_range"] ** 2
    df["temp_extreme_high"] = (df["avg_temperature"] > 25).astype(int)
    df["temp_extreme_low"] = (df["avg_temperature"] < 18).astype(int)
    df["temp_optimal"] = ((df["avg_temperature"] >= 20) & (df["avg_temperature"] <= 25)).astype(int)
    
    # Temperature anomalies
    df["temp_anomaly_30d"] = df["avg_temperature"] - df["tavg_30d"]
    df["temp_anomaly_7d"] = df["avg_temperature"] - df["tavg_7d"]
    df["temp_anomaly_90d"] = df["avg_temperature"] - df["tavg_90d"]
    df["temp_anomaly_abs"] = df["temp_anomaly_30d"].abs()
    
    # Temperature extremes
    df["max_temp_extreme"] = (df["max_temperature"] > 30).astype(int)
    df["min_temp_extreme"] = (df["min_temperature"] < 15).astype(int)
    df["max_temp_high"] = (df["max_temperature"] > 28).astype(int)
    df["min_temp_low"] = (df["min_temperature"] < 18).astype(int)
    
    # Temperature variability
    df["temp_variability"] = df["temp_range_mean_30d"] / (df["avg_temperature"] + 1)
    df["temp_stability"] = 1 / (df["temp_range_mean_30d"] + 0.1)
    df["diurnal_range_ratio"] = df["temperature_range"] / (df["temp_range_mean_30d"] + 0.1)
    
    # Temperature trends
    df["temp_trend_7d"] = df["tavg_7d"] - df["tavg_30d"]
    df["temp_trend_90d"] = df["tavg_90d"] - df["tavg_30d"]
    df["temp_warming"] = (df["temp_trend_7d"] > 2).astype(int)
    df["temp_cooling"] = (df["temp_trend_7d"] < -2).astype(int)
    
    # ========== PRECIPITATION FEATURES ==========
    df["is_heavy_rain"] = (df["precipitation"] > 5).astype(int)
    df["is_extreme_rain"] = (df["precipitation"] > 10).astype(int)
    df["is_rainy_day"] = (df["precipitation"] > 0.1).astype(int)
    df["is_light_rain"] = ((df["precipitation"] > 0.1) & (df["precipitation"] <= 2)).astype(int)
    
    # Rain intensity
    df["rain_intensity"] = df["precipitation"] / (df["rain_days_30d"] + 1)
    df["rain_intensity_squared"] = df["rain_intensity"] ** 2
    
    # Rain anomalies
    df["rain_anomaly"] = df["rain_sum_7d"] - (df["rain_sum_30d"] / 30 * 7)
    df["rain_anomaly_pct"] = df["rain_anomaly"] / (df["rain_sum_30d"] / 30 * 7 + 0.1)
    df["rain_anomaly_abs"] = df["rain_anomaly"].abs()
    
    # Rain ratios
    df["recent_rain_ratio"] = df["rain_sum_7d"] / (df["rain_sum_30d"] + 1)
    df["recent_rain_ratio_90d"] = df["rain_sum_30d"] / (df["rain_sum_90d"] + 1)
    df["rain_frequency"] = df["rain_days_30d"] / 30
    
    # Prolonged conditions
    df["prolonged_rain"] = (df["rain_days_30d"] > 20).astype(int)
    df["drought_period"] = (df["rain_days_30d"] < 5).astype(int)
    df["heavy_rain_event"] = (df["max_daily_rain_30d"] > 20).astype(int)
    
    # ========== NDVI (VEGETATION) FEATURES ==========
    df["ndvi_change"] = df["ndvi_30d"] - df["ndvi_90d"]
    df["ndvi_change_pct"] = df["ndvi_change"] / (df["ndvi_90d"] + 0.01)
    df["ndvi_high"] = (df["ndvi_30d"] > 0.7).astype(int)
    df["ndvi_low"] = (df["ndvi_30d"] < 0.4).astype(int)
    df["ndvi_moderate"] = ((df["ndvi_30d"] >= 0.4) & (df["ndvi_30d"] <= 0.7)).astype(int)
    df["ndvi_declining"] = (df["ndvi_change"] < -0.1).astype(int)
    df["ndvi_improving"] = (df["ndvi_change"] > 0.1).astype(int)
    
    # Vegetation stress
    df["vegetation_stress"] = ((df["ndvi_30d"] < 0.5) & (df["avg_temperature"] > 23)).astype(int)
    df["vegetation_health"] = ((df["ndvi_30d"] > 0.6) & (df["precipitation"] > 1)).astype(int)
    
    # ========== ELEVATION/TERRAIN FEATURES ==========
    df["high_elevation"] = (df["elevation"] > 1500).astype(int)
    df["very_high_elevation"] = (df["elevation"] > 2000).astype(int)
    df["low_elevation"] = (df["elevation"] < 1000).astype(int)
    df["steep_slope"] = (df["slope"] > 5).astype(int)
    df["very_steep_slope"] = (df["slope"] > 10).astype(int)
    df["gentle_slope"] = (df["slope"] < 1).astype(int)
    df["elevation_log"] = np.log1p(df["elevation"])
    df["slope_log"] = np.log1p(df["slope"])
    
    # Elevation interactions
    df["elevation_temp_interaction"] = df["elevation"] * df["avg_temperature"] / 1000
    df["elevation_rain_interaction"] = df["elevation"] * df["precipitation"] / 1000
    df["high_elev_cold"] = df["high_elevation"] * df["temp_extreme_low"]
    df["slope_rain"] = df["steep_slope"] * df["is_heavy_rain"]
    
    # ========== HOT/COLD DAY FEATURES ==========
    df["heat_wave"] = (df["hot_days_30d"] > 5).astype(int)
    df["extreme_heat"] = (df["hot_days_30d"] > 10).astype(int)
    df["severe_heat"] = (df["hot_days_30d"] > 15).astype(int)
    df["heat_days_ratio"] = df["hot_days_30d"] / 30
    df["heat_frequency"] = np.log1p(df["hot_days_30d"])
    
    # ========== INTERACTION FEATURES ==========
    # Age × Climate interactions
    df["elderly_heat"] = df["is_elderly"] * df["temp_extreme_high"]
    df["elderly_cold"] = df["is_elderly"] * df["temp_extreme_low"]
    df["child_rain"] = df["is_child"] * df["is_rainy_day"]
    df["child_heat"] = df["is_child"] * df["heat_wave"]
    df["vulnerable_extreme"] = df["is_vulnerable_age"] * df["temp_extreme_high"]
    df["vulnerable_rain"] = df["is_vulnerable_age"] * df["is_heavy_rain"]
    df["infant_temp_stress"] = df["is_infant"] * df["temp_anomaly_abs"]
    
    # Rain × Temperature interactions
    df["hot_humid"] = df["temp_extreme_high"] * df["is_rainy_day"]
    df["hot_dry"] = df["temp_extreme_high"] * df["drought_period"]
    df["cold_wet"] = df["temp_extreme_low"] * df["is_heavy_rain"]
    df["comfort_index"] = df["avg_temperature"] * (1 - df["rain_frequency"])
    
    # NDVI × Temperature interactions
    df["vegetation_heat"] = df["ndvi_high"] * df["temp_extreme_high"]
    df["vegetation_drought"] = df["ndvi_low"] * df["drought_period"]
    
    # Multi-way interactions
    df["triple_risk"] = df["is_elderly"] * df["heat_wave"] * df["drought_period"]
    df["compound_stress"] = df["temp_extreme_high"] + df["is_heavy_rain"] + df["heat_wave"]
    
    # ========== COMPOSITE RISK SCORES ==========
    df["climate_stress_score"] = (
        df["hot_days_30d"] / 10 +
        df["temp_anomaly_abs"] * 2 +
        df["rain_anomaly_abs"] / 10 +
        df["is_vulnerable_age"] * 2 +
        df["heat_wave"] * 1.5
    )
    
    df["environmental_risk"] = (
        df["temp_extreme_high"].astype(float) +
        df["is_heavy_rain"].astype(float) +
        df["ndvi_low"].astype(float) +
        df["high_elevation"].astype(float) +
        df["heat_wave"].astype(float) * 0.5
    )
    
    df["health_vulnerability_index"] = (
        df["is_vulnerable_age"] * 3 +
        df["climate_stress_score"] +
        df["compound_stress"] * 0.5
    )
    
    # ========== LOCATION FEATURES ==========
    df["lat_bin"] = np.digitize(df["latitude"], bins=[0.3, 0.6, 1.0, 1.5])
    df["lon_bin"] = np.digitize(df["longitude"], bins=[31, 32.5, 33.5, 34.5])
    df["dist_from_equator"] = df["latitude"].abs()
    df["geo_risk"] = df["lat_bin"] + df["lon_bin"]
    
    # ========== GENDER/ZONE FEATURES ==========
    df["is_male"] = (df["gender"] == "Male").astype(int)
    df["is_female"] = (df["gender"] == "Female").astype(int)
    df["is_rural"] = (df["zone"] == "Rural").astype(int)
    df["is_urban"] = (df["zone"].isin(["Urban", "Peri_urban"])).astype(int)
    df["is_peri_urban"] = (df["zone"] == "Peri_urban").astype(int)
    
    # Gender × Environment interactions
    df["male_rural"] = df["is_male"] * df["is_rural"]
    df["female_rural"] = df["is_female"] * df["is_rural"]
    df["male_heat"] = df["is_male"] * df["heat_wave"]
    df["female_heat"] = df["is_female"] * df["heat_wave"]
    
    # Drop raw date
    df = df.drop(columns=["deathdate"], errors='ignore')
    
    return df

# Apply feature engineering
print("=" * 60)
print("DEEP FEATURE ENGINEERING")
print("=" * 60)

train = deep_feature_engineering(train)
test = deep_feature_engineering(test)

print(f"Original features: ~27")
print(f"Features after engineering: {train.shape[1] - 2}")
print(f"New features created: ~{train.shape[1] - 29}")

## 📊 Feature Selection & Preprocessing

In [ ]:
# Identify feature columns
exclude_cols = [ID_COL, TARGET, "location", "age_group", "lat_bin", "lon_bin"]
feature_cols = [c for c in train.columns if c not in exclude_cols]

X = train[feature_cols].copy()
y = train[TARGET].copy()
X_test = test[feature_cols].copy()

# Categorical and numeric columns
cat_cols = ['zone', 'gender']
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

# Remove highly correlated features (optional)
def remove_highly_correlated(X, threshold=0.95):
    """Remove features with correlation > threshold"""
    corr_matrix = X.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    print(f"Removing {len(to_drop)} highly correlated features: {to_drop[:10]}...")
    return X.drop(columns=to_drop), to_drop

# Apply correlation filter
X_filtered, dropped = remove_highly_correlated(X[num_cols], threshold=0.95)
num_cols_filtered = list(X_filtered.columns)
feature_cols_filtered = num_cols_filtered + cat_cols

X = X[feature_cols_filtered]
X_test = X_test[feature_cols_filtered]

print(f"Features after correlation filter: {len(feature_cols_filtered)}")

## 🔧 Advanced Preprocessing Pipeline

In [ ]:
# Advanced preprocessing with KNN imputation
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
        ("num", Pipeline([
            ("imputer", KNNImputer(n_neighbors=5)),
            ("scaler", RobustScaler())
        ]), num_cols_filtered)
    ]
)

print("Preprocessor configured:")
print(f"  - Categorical: KNN Imputer + OneHotEncoding")
print(f"  - Numeric: KNN Imputer (k=5) + RobustScaler")

## 🤖 Advanced Model Definitions

In [ ]:
def get_advanced_models():
    """Return dictionary of optimized models"""
    models = {}
    
    # XGBoost with optimized parameters
    if HAS_XGB:
        models['XGBoost'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", xgb.XGBClassifier(
                n_estimators=300,
                max_depth=7,
                learning_rate=0.03,
                subsample=0.8,
                colsample_bytree=0.8,
                colsample_bylevel=0.8,
                min_child_weight=3,
                gamma=0.1,
                reg_alpha=0.1,
                reg_lambda=1.0,
                scale_pos_weight=1.5,
                random_state=RANDOM_STATE,
                eval_metric='auc',
                tree_method='hist',
                n_jobs=-1
            ))
        ])
        
        models['XGBoost_Focal'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", xgb.XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                min_child_weight=2,
                scale_pos_weight=2.0,
                random_state=RANDOM_STATE,
                eval_metric='auc',
                tree_method='hist',
                n_jobs=-1
            ))
        ])
    
    # LightGBM with optimized parameters
    if HAS_LGBM:
        models['LightGBM'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", lgb.LGBMClassifier(
                n_estimators=300,
                max_depth=8,
                learning_rate=0.03,
                num_leaves=31,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_samples=20,
                reg_alpha=0.1,
                reg_lambda=1.0,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1,
                n_jobs=-1
            ))
        ])
        
        models['LightGBM_Large'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", lgb.LGBMClassifier(
                n_estimators=500,
                max_depth=10,
                learning_rate=0.02,
                num_leaves=50,
                subsample=0.85,
                colsample_bytree=0.85,
                min_child_samples=15,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1,
                n_jobs=-1
            ))
        ])
    
    # CatBoost
    if HAS_CAT:
        models['CatBoost'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", cb.CatBoostClassifier(
                iterations=300,
                depth=7,
                learning_rate=0.03,
                l2_leaf_reg=3.0,
                class_weights="balanced",
                random_state=RANDOM_STATE,
                verbose=0,
                thread_count=-1
            ))
        ])
    
    # Random Forest
    models['RandomForest'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    
    # Extra Trees
    models['ExtraTrees'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", ExtraTreesClassifier(
            n_estimators=300,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    
    # Gradient Boosting
    models['GradientBoosting'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            min_samples_split=10,
            min_samples_leaf=5,
            subsample=0.8,
            random_state=RANDOM_STATE
        ))
    ])
    
    # HistGradientBoosting (fast sklearn GBM)
    models['HistGB'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", HistGradientBoostingClassifier(
            max_iter=300,
            max_depth=8,
            learning_rate=0.05,
            l2_regularization=0.1,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])
    
    # Calibrated Logistic Regression
    models['LogisticRegression'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", CalibratedClassifierCV(
            LogisticRegression(
                max_iter=2000,
                C=0.5,
                class_weight="balanced",
                solver="lbfgs",
                random_state=RANDOM_STATE
            ),
            method="isotonic",
            cv=5
        ))
    ])
    
    # MLP Neural Network
    models['MLP'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", MLPClassifier(
            hidden_layer_sizes=(128, 64, 32),
            activation='relu',
            solver='adam',
            alpha=0.01,
            batch_size=32,
            learning_rate='adaptive',
            max_iter=500,
            random_state=RANDOM_STATE,
            early_stopping=True,
            validation_fraction=0.1
        ))
    ])
    
    return models

models = get_advanced_models()
print(f"\nModels configured: {list(models.keys())}")
print(f"Number of models: {len(models)}")

## 📈 Cross-Validation Evaluation

In [ ]:
print("=" * 60)
print("CROSS-VALIDATION EVALUATION")
print("=" * 60)

cv_results = {}
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    
    try:
        # Get probability predictions via cross-validation
        cv_proba = cross_val_predict(model, X, y, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
        cv_pred = (cv_proba >= 0.5).astype(int)
        
        # Calculate metrics
        f1 = f1_score(y, cv_pred)
        auc = roc_auc_score(y, cv_proba)
        final_score = 0.60 * f1 + 0.40 * auc
        
        cv_results[name] = {
            'model': model,
            'f1': f1,
            'auc': auc,
            'final_score': final_score,
            'cv_proba': cv_proba
        }
        
        print(f"  F1 Score:    {f1:.4f}")
        print(f"  ROC-AUC:     {auc:.4f}")
        print(f"  Final Score: {final_score:.4f}")
    except Exception as e:
        print(f"  Error: {e}")

# Display results table
print("\n" + "=" * 60)
print("CV RESULTS SUMMARY")
print("=" * 60)

if cv_results:
    results_df = pd.DataFrame({
        'Model': list(cv_results.keys()),
        'F1 Score': [cv_results[m]['f1'] for m in cv_results],
        'ROC-AUC': [cv_results[m]['auc'] for m in cv_results],
        'Final Score': [cv_results[m]['final_score'] for m in cv_results]
    }).sort_values('Final Score', ascending=False)
    
    print(results_df.to_string(index=False))
else:
    print("No models evaluated successfully!")

## 🎯 Threshold Optimization

In [ ]:
print("=" * 60)
print("THRESHOLD OPTIMIZATION")
print("=" * 60)

if not cv_results:
    print("No results to optimize!")
else:
    # Find best model
    best_model_name = max(cv_results, key=lambda x: cv_results[x]['final_score'])
    print(f"Best model: {best_model_name}")
    
    # Optimize threshold for best model
    best_proba = cv_results[best_model_name]['cv_proba']
    
    def objective(threshold):
        pred = (best_proba >= threshold).astype(int)
        f1_t = f1_score(y, pred)
        auc_t = roc_auc_score(y, best_proba)
        return -(0.60 * f1_t + 0.40 * auc_t)  # Minimize negative
    
    from scipy.optimize import minimize_scalar
    result = minimize_scalar(objective, bounds=(0.3, 0.7), method='bounded')
    best_threshold = result.x
    best_score = -result.fun
    
    # Calculate metrics at optimal threshold
    optimal_pred = (best_proba >= best_threshold).astype(int)
    optimal_f1 = f1_score(y, optimal_pred)
    
    print(f"\nOptimal threshold: {best_threshold:.4f}")
    print(f"F1 at optimal: {optimal_f1:.4f}")
    print(f"Best combined score: {best_score:.4f}")
    
    # Plot threshold analysis
    thresholds = np.arange(0.30, 0.70, 0.01)
    threshold_results = []
    for thresh in thresholds:
        pred = (best_proba >= thresh).astype(int)
        f1_t = f1_score(y, pred)
        combined = 0.60 * f1_t + 0.40 * roc_auc_score(y, best_proba)
        threshold_results.append({'threshold': thresh, 'f1': f1_t, 'combined': combined})
    
    thresh_df = pd.DataFrame(threshold_results)
    
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(thresh_df['threshold'], thresh_df['f1'], 'b-', linewidth=2, label='F1 Score')
    plt.axvline(best_threshold, color='r', linestyle='--', linewidth=2, label=f'Optimal: {best_threshold:.3f}')
    plt.xlabel('Threshold')
    plt.ylabel('F1 Score')
    plt.title('F1 Score vs Threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(thresh_df['threshold'], thresh_df['combined'], 'g-', linewidth=2, label='Combined Score')
    plt.axvline(best_threshold, color='r', linestyle='--', linewidth=2, label=f'Optimal: {best_threshold:.3f}')
    plt.xlabel('Threshold')
    plt.ylabel('Combined Score')
    plt.title('Combined Score vs Threshold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 🏆 Stacking Ensemble

In [ ]:
print("=" * 60)
print("STACKING ENSEMBLE")
print("=" * 60)

if not cv_results:
    print("No models available for stacking!")
else:
    # Select top models for stacking
    top_models = sorted(cv_results.keys(), key=lambda x: cv_results[x]['final_score'], reverse=True)[:5]
    print(f"Top models for stacking: {top_models}")
    
    # Create stacking ensemble
    base_models = [(name, cv_results[name]['model']) for name in top_models]
    
    # Meta-learner
    meta_learner = LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
    
    stacking_clf = StackingClassifier(
        estimators=base_models,
        final_estimator=meta_learner,
        cv=5,
        stack_method="predict_proba",
        n_jobs=-1
    )
    
    # Evaluate stacking ensemble
    print("\nEvaluating Stacking Ensemble...")
    stacking_proba = cross_val_predict(stacking_clf, X, y, cv=skf, method="predict_proba", n_jobs=-1)[:, 1]
    stacking_pred = (stacking_proba >= best_threshold).astype(int)
    
    stacking_f1 = f1_score(y, stacking_pred)
    stacking_auc = roc_auc_score(y, stacking_proba)
    stacking_final = 0.60 * stacking_f1 + 0.40 * stacking_auc
    
    print(f"\nStacking Ensemble Results:")
    print(f"  F1 Score:    {stacking_f1:.4f}")
    print(f"  ROC-AUC:     {stacking_auc:.4f}")
    print(f"  Final Score: {stacking_final:.4f}")
    
    cv_results['Stacking'] = {
        'model': stacking_clf,
        'f1': stacking_f1,
        'auc': stacking_auc,
        'final_score': stacking_final,
        'cv_proba': stacking_proba
    }
    
    # Update results
    results_df = pd.DataFrame({
        'Model': list(cv_results.keys()),
        'F1 Score': [cv_results[m]['f1'] for m in cv_results],
        'ROC-AUC': [cv_results[m]['auc'] for m in cv_results],
        'Final Score': [cv_results[m]['final_score'] for m in cv_results]
    }).sort_values('Final Score', ascending=False)
    
    print("\n" + "=" * 60)
    print("UPDATED RESULTS (with Stacking)")
    print("=" * 60)
    print(results_df.to_string(index=False))

## 🎲 Weighted Ensemble Predictions

In [ ]:
print("=" * 60)
print("ENSEMBLE PREDICTIONS")
print("=" * 60)

if not cv_results:
    print("No models to ensemble!")
else:
    # Calculate model weights based on CV performance
    total_score = sum(cv_results[m]['final_score'] for m in cv_results)
    model_weights = {m: cv_results[m]['final_score'] / total_score for m in cv_results}
    
    print("Model weights for weighted ensemble:")
    for m, w in sorted(model_weights.items(), key=lambda x: -x[1]):
        print(f"  {m:25s}: {w:.4f}")
    
    # Train all models and create weighted ensemble
    ensemble_proba = np.zeros(len(X_test))
    
    for name, model in models.items():
        if name in cv_results:
            print(f"Training {name}...")
            model.fit(X, y)
            test_proba = model.predict_proba(X_test)[:, 1]
            ensemble_proba += model_weights[name] * test_proba
    
    # Also compute simple average ensemble
    all_test_probas = []
    for name, model in models.items():
        if name in cv_results:
            model.fit(X, y)
            test_proba = model.predict_proba(X_test)[:, 1]
            all_test_probas.append(test_proba)
    
    simple_avg = np.mean(all_test_probas, axis=0)
    
    # Blend weighted ensemble with simple average and stacking
    if 'Stacking' in cv_results:
        # Train stacking model
        print("Training Stacking Ensemble...")
        stacking_clf.fit(X, y)
        stacking_test_proba = stacking_clf.predict_proba(X_test)[:, 1]
        
        # Final blend: 50% weighted, 30% simple avg, 20% stacking
        final_proba = 0.5 * ensemble_proba + 0.3 * simple_avg + 0.2 * stacking_test_proba
    else:
        final_proba = 0.7 * ensemble_proba + 0.3 * simple_avg
    
    # Apply optimal threshold
    final_pred = (final_proba >= best_threshold).astype(int)
    
    print(f"\nEnsemble probability range: [{final_proba.min():.3f}, {final_proba.max():.3f}]")
    print(f"Predicted positives: {final_pred.sum()} / {len(final_pred)} ({100*final_pred.mean():.1f}%)")

## 📊 Feature Importance Analysis

In [ ]:
print("=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

if not cv_results:
    print("No models to analyze!")
else:
    # Use best tree-based model for feature importance
    best_model_name = max(cv_results, key=lambda x: cv_results[x]['final_score'])
    if best_model_name == 'Stacking':
        best_model_name = list(cv_results.keys())[0]
    
    best_model = models[best_model_name]
    best_model.fit(X, y)
    
    try:
        trained_model = best_model.named_steps['model']
        
        if hasattr(trained_model, 'feature_importances_'):
            importances = trained_model.feature_importances_
            
            # Get feature names
            ohe = best_model.named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot']
            cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
            feature_names = cat_feature_names + num_cols_filtered
            
            imp_df = pd.DataFrame({
                'feature': feature_names,
                'importance': importances
            }).sort_values('importance', ascending=False)
            
            # Plot top 25 features
            plt.figure(figsize=(10, 10))
            top_features = imp_df.head(25)
            plt.barh(range(len(top_features)), top_features['importance'].values)
            plt.yticks(range(len(top_features)), top_features['feature'].values)
            plt.xlabel('Importance')
            plt.title(f'Top 25 Feature Importances ({best_model_name})')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.show()
            
            print("\nTop 20 features:")
            for i, row in imp_df.head(20).iterrows():
                print(f"  {row['feature']:35s}: {row['importance']:.4f}")
    except Exception as e:
        print(f"Could not extract feature importance: {e}")

## 📝 Create Submission File

In [ ]:
print("=" * 60)
print("CREATING SUBMISSION")
print("=" * 60)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    "TargetF1": final_pred,
    "TargetRAUC": final_proba
})

submission.to_csv("champion_submission.csv", index=False)

print(f"Submission saved: champion_submission.csv")
print(f"Shape: {submission.shape}")
print(f"\nTargetF1 distribution:")
print(submission['TargetF1'].value_counts())
print(f"\nTargetRAUC statistics:")
print(submission['TargetRAUC'].describe())

## 📋 Final Summary

In [ ]:
print("=" * 60)
print("🏆 CHAMPION MODEL SUMMARY")
print("=" * 60)

if cv_results:
    best_model_name = max(cv_results, key=lambda x: cv_results[x]['final_score'])
    
    print("\n📊 Cross-Validation Results:")
    print("-" * 60)
    for name in results_df['Model']:
        r = cv_results[name]
        print(f"{name:25s} | F1: {r['f1']:.4f} | AUC: {r['auc']:.4f} | Score: {r['final_score']:.4f}")
    
    print("\n" + "=" * 60)
    print("BEST MODEL CONFIGURATION")
    print("=" * 60)
    print(f"Best Model:          {best_model_name}")
    print(f"Optimal Threshold:   {best_threshold:.4f}")
    print(f"CV F1 Score:         {cv_results[best_model_name]['f1']:.4f}")
    print(f"CV ROC-AUC:          {cv_results[best_model_name]['auc']:.4f}")
    print(f"CV Final Score:      {cv_results[best_model_name]['final_score']:.4f}")
    
    if 'Stacking' in cv_results:
        print(f"\n🏆 Stacking Score:    {cv_results['Stacking']['final_score']:.4f}")
    
    print(f"\n📁 Submission file:    champion_submission.csv")
    print(f"📊 Predicted positives: {final_pred.sum()} ({100*final_pred.mean():.1f}%)")
    
    print("\n" + "=" * 60)
    print("✅ KEY IMPROVEMENTS")
    print("=" * 60)
    print("""
1. DEEP FEATURE ENGINEERING (80+ features):
   - Cyclical temporal features (day, month, week, season)
   - Detailed age groups (infant, toddler, teen, elderly, etc.)
   - Temperature anomalies, trends, and variability
   - Precipitation intensity, frequency, and anomalies
   - NDVI change, vegetation stress indicators
   - Elevation and slope interactions
   - Multi-way interaction terms (age×climate, rain×temp)
   - Composite risk scores

2. ADVANCED MODELS:
   - XGBoost (2 variants with tuned parameters)
   - LightGBM (2 variants including large model)
   - CatBoost with depth optimization
   - RandomForest & ExtraTrees (deep trees)
   - HistGradientBoosting (fast GBM)
   - Calibrated Logistic Regression
   - MLP Neural Network (128-64-32)

3. ENSEMBLE STRATEGIES:
   - Stacking with Logistic Regression meta-learner
   - Weighted ensemble based on CV performance
   - Blend: 50% weighted + 30% simple avg + 20% stacking

4. OPTIMIZATION:
   - Threshold optimization using scipy minimize
   - Correlation-based feature selection
   - KNN imputation for missing values
   - RobustScaler for outlier resistance
    """)
else:
    print("No results to display!")